# Module 2 — Prompt Engineering: Hands-On Lab Notebook
## Professional Generative AI Applications Certification
### Sessions 6–10 | UpSkill Global Education Technologies Inc., Canada

---

| Session | Topic | Key Labs |
|---------|-------|----------|
| 6 | Prompt Engineering Fundamentals & CRAFT Framework | CRAFT builder, quality scoring |
| 7 | Zero-Shot, One-Shot & Few-Shot Prompting | Classification, style replication |
| 8 | Chain-of-Thought & Advanced Prompting | CoT, self-consistency, self-critique |
| 9 | Persona Prompting & Role Engineering | Persona engineering, multi-persona, system prompts |
| 10 | Prompt Library & Optimization | Optimization cycle, prompt library builder |

**Prerequisites:** `pip install openai python-dotenv pandas`

In [ ]:
# ============================================================
# GLOBAL SETUP — Run this cell before any lab exercise
# ============================================================
import os, json, time, datetime
from IPython.display import Markdown, display
from openai import OpenAI

API_KEY = "sk-your-api-key-here"
client = OpenAI(api_key=API_KEY)

def ask(prompt, model='gpt-4o-mini', temperature=0.7, max_tokens=800, system=None):
    messages = []
    if system: messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})
    r = client.chat.completions.create(model=model, messages=messages, temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content

def show(text, title=None):
    if title: display(Markdown(f'### {title}'))
    display(Markdown(text))

def craft_prompt(context, role, action, fmt, tone, constraints=None):
    """Assemble a CRAFT-structured prompt."""
    p = f'CONTEXT: {context}\n\nROLE: {role}\n\nACTION: {action}\n\nFORMAT: {fmt}\n\nTONE: {tone}'
    if constraints: p += f'\n\nCONSTRAINTS: {constraints}'
    return p

print('Setup complete.')

---
# SESSION 6 — Prompt Engineering Fundamentals & CRAFT Framework
## Labs: CRAFT Builder, Weak vs Strong Prompts, Quality Scoring

### Lab 6.1 — The CRAFT Framework Builder
**Task:** Build prompts systematically using Context, Role, Action, Format, Tone (+ Constraints).

In [ ]:
# Lab 6.1 — CRAFT Framework in Action

# Example 1: HR Announcement
p1 = craft_prompt(
    context='Our company (400 employees, IT services) is introducing a 4-day work week pilot for Q1 2025.',
    role='You are the Chief People Officer drafting the internal announcement.',
    action='Write the company-wide announcement email for the 4-day work week pilot.',
    fmt='Email format: Subject line + 3 paragraphs + sign-off. Max 200 words.',
    tone='Warm, inclusive, transparent. Acknowledge both excitement and practical concerns.',
    constraints='Do not promise the pilot will become permanent. Include pilot dates and feedback mechanism.'
)
show(ask(p1, temperature=0.5), 'CRAFT Output 1 — HR Announcement')

print('\n' + '=' * 60 + '\n')

# Example 2: Customer Apology
p2 = craft_prompt(
    context='Our SaaS platform had a 6-hour outage affecting 3,000 customers. Root cause: failed database migration.',
    role='You are VP of Customer Success writing a public incident communication.',
    action='Write the customer apology email explaining what happened and what we are doing about it.',
    fmt='Email: Subject + 4 paragraphs (apology, what happened, what we did, prevention). Max 200 words.',
    tone='Accountable, sincere, action-oriented. No corporate deflection.',
    constraints='Do NOT use: "We apologize for any inconvenience." Be specific about cause and fix.'
)
show(ask(p2, temperature=0.4), 'CRAFT Output 2 — Customer Apology')

### Lab 6.2 — Weak vs Strong Prompt Comparison
**Task:** Compare outputs from vague prompts vs CRAFT-structured prompts.

In [ ]:
# Lab 6.2 — Weak vs Strong Prompt Comparison
weak = 'Write a marketing email for our new product.'
strong = craft_prompt(
    context='We are launching an AI-powered inventory tool for small retailers in India. Price: 4,999/month. Works with existing POS.',
    role='You are a senior B2B email copywriter with SaaS experience.',
    action='Write a cold outreach email to retail store owners introducing our product.',
    fmt='Subject line + 3 paragraphs (pain point, solution, CTA). Under 150 words. CTA: book a 15-min demo.',
    tone='Direct, benefit-focused, conversational. Not salesy.',
    constraints='Do NOT use: revolutionary, game-changing, synergy. Lead with their problem, not our product.'
)

print('WEAK PROMPT OUTPUT:')
print(f'Prompt: "{weak}"\n')
print(ask(weak, temperature=0.5))
print('\n' + '=' * 60)
print('\nSTRONG (CRAFT) PROMPT OUTPUT:')
print(ask(strong, temperature=0.4))

print('\nREFLECTION: How many iterations of the weak prompt would it take to match the strong prompt\'s first output?')

### Lab 6.3 — CRAFT Quality Scoring
**Task:** Use AI to audit and score prompts against the CRAFT framework.

In [ ]:
# Lab 6.3 — CRAFT Quality Audit
prompts_to_audit = [
    'Write about digital transformation.',
    'You are a consultant. Write a report about AI adoption for my company.',
    craft_prompt(
        context='A 200-person manufacturing company considering AI for quality control.',
        role='You are a senior technology consultant specializing in manufacturing AI.',
        action='Write a 1-page recommendation on whether to implement AI-based visual inspection.',
        fmt='3 sections: Recommendation (2 sentences), Business Case (3 bullets with ROI), Implementation Roadmap (3 milestones). Under 300 words.',
        tone='Professional, evidence-based, practical. Written for a CFO.',
        constraints='Do not recommend specific vendors. Focus on the decision framework.'
    )
]

for i, prompt_text in enumerate(prompts_to_audit, 1):
    audit = (
        f'You are a prompt engineering expert. Score this prompt on CRAFT (each 1-5):\n'
        f'C (Context) | R (Role) | A (Action) | F (Format) | T (Tone) | Constraints\n'
        f'Markdown table with Score + 1-sentence reason. Total /30.\n'
        f'Rating: Excellent (26-30) / Good (20-25) / Needs Work (14-19) / Poor (<14)\n\n'
        f'Prompt: "{prompt_text}"'
    )
    show(ask(audit, temperature=0.2), f'CRAFT Audit — Prompt {i}')
    print()

---
# SESSION 7 — Zero-Shot, One-Shot & Few-Shot Prompting
## Labs: Classification, Style Replication, Shot-Type Comparison

### Lab 7.1 — Zero-Shot Classification & Extraction
**Task:** Classify and extract data without providing any examples.

In [ ]:
# Lab 7.1 — Zero-Shot Sentiment + Entity Extraction
reviews = [
    'The delivery was late but product quality exceeded my expectations.',
    'Absolutely terrible experience. Never ordering again.',
    'It was okay. Nothing special but did the job.',
    'Amazing product! But customer service was rude when I had a question.',
    'Perfect in every way. Will recommend to everyone I know.',
    'The product broke after 2 days. Refund process was smooth though.',
]

print('ZERO-SHOT SENTIMENT CLASSIFICATION')
print('=' * 70)
for r in reviews:
    result = ask(
        f'Classify as Positive, Negative, or Mixed. ONLY output: label + 5-word reason.\nReview: "{r}"',
        temperature=0.0
    )
    print(f'  Review: {r[:60]}...')
    print(f'  => {result}\n')

# Zero-shot entity extraction
print('ZERO-SHOT CONTRACT EXTRACTION')
print('=' * 70)
contract = (
    'Agreement dated 15 March 2024 between TechFlow Pvt Ltd (Mumbai) and '
    'Nexus Retail Ltd (Delhi). Cloud services for 24 months. '
    'Fee: INR 85,000/month + GST. Payment: Net 30. Termination: 90 days notice.'
)
schema = '{"parties":[], "date":"", "duration":"", "fee":"", "payment_terms":"", "termination":""}'
print(ask(f'Extract into JSON: {schema}\nContract: {contract}', temperature=0.0))

### Lab 7.2 — Few-Shot Style Replication
**Task:** Provide 2 examples to teach AI a specific writing style, then generate new ones.

In [ ]:
# Lab 7.2 — Few-Shot Customer Support Response
few_shot = (
    'You write customer support responses with this exact pattern:\n'
    '(1) Acknowledge specific problem (2) One apology (3) Exact resolution + timeline (4) Close with name\n\n'
    'EXAMPLE 1:\nCOMPLAINT: Order #4521 arrived damaged — screen cracked.\n'
    'RESPONSE: Your order #4521 arrived with a cracked screen — that should never happen. '
    'I sincerely apologize. I have arranged a replacement within 48 hours at no cost. — Priya\n\n'
    'EXAMPLE 2:\nCOMPLAINT: Charged twice for subscription. Rs 2,999 appeared twice.\n'
    'RESPONSE: You were double-charged Rs 2,999 — our billing error, I apologize. '
    'Full refund initiated; it will reflect in 5-7 business days. — Rahul\n\n'
    'Now respond to these NEW complaints:\n\n'
    'COMPLAINT 3: Waited 3 weeks for laptop repair. No updates. Need it for work.\nCUSTOMER: Anjali\n\n'
    'COMPLAINT 4: Wrong color delivered. Ordered black, received white.\nCUSTOMER: Vikram\n\n'
    'COMPLAINT 5: App crashes after latest update. Cannot access account.\nCUSTOMER: Meera'
)

show(ask(few_shot, temperature=0.4), 'Few-Shot Responses')

print('\nEVALUATION CHECKLIST:')
for check in ['Acknowledges SPECIFIC problem?', 'Exactly ONE apology?', 'Exact resolution + TIMELINE?', 'Closes with NAME?', 'Matches example tone and length?']:
    print(f'  [ ] {check}')

### Lab 7.3 — Zero-Shot vs Few-Shot Comparison
**Task:** Compare output quality with and without examples.

In [ ]:
# Lab 7.3 — Zero-Shot vs Few-Shot: Email Subject Lines
zero_shot = 'Write 5 email subject lines for a B2B cold outreach about our AI analytics platform. Curiosity-driven, under 50 chars.'

few_shot = (
    'Write 5 email subject lines for a B2B cold outreach about our AI analytics platform. Curiosity-driven, under 50 chars.\n\n'
    'Style examples:\n'
    '- "Your dashboards are lying to you"\n'
    '- "What if reports wrote themselves?"\n'
    '- "3 hrs/week on reports? Let\'s fix that"\n\n'
    'Write 5 NEW ones in the same style (different angles).'
)

print('ZERO-SHOT:'); print(ask(zero_shot, temperature=0.7))
print('\n' + '=' * 60)
print('\nFEW-SHOT:'); print(ask(few_shot, temperature=0.7))
print('\nREFLECTION: Few-shot gives AI a "voice" to match. Zero-shot uses generic patterns.')

---
# SESSION 8 — Chain-of-Thought & Advanced Prompting
## Labs: CoT Reasoning, Self-Consistency, Self-Critique, Decomposition

### Lab 8.1 — Direct Answer vs Chain-of-Thought
**Task:** Compare a direct answer with step-by-step reasoning.

In [ ]:
# Lab 8.1 — Direct vs Chain-of-Thought
scenario = (
    'FMCG company considering premium organic snack launch.\n'
    'Revenue: 120Cr. Organic CAGR: 28%. Manufacturing cost: +40%. Price: Rs 80 vs Rs 30.\n'
    'Marketing: 8Cr Year 1. Competitor: 14-month head start.'
)

print('DIRECT ANSWER:')
print(ask(f'{scenario}\n\nShould they launch? 2 sentences.', temperature=0.3))

cot = (
    f'{scenario}\n\nThink step by step:\n'
    'Step 1: Market opportunity (size, growth, timing)\n'
    'Step 2: Financial viability (margins, break-even, investment)\n'
    'Step 3: Competitive risk (head start, differentiation)\n'
    'Step 4: Recommendation with ONE critical condition'
)
print('\nCHAIN-OF-THOUGHT:')
show(ask(cot, temperature=0.3, max_tokens=800))
print('\nREFLECTION: CoT forces AI to show reasoning — making errors visible and logic verifiable.')

### Lab 8.2 — Self-Consistency Check
**Task:** Run the same CoT prompt 3 times — do all runs reach the same answer?

In [ ]:
# Lab 8.2 — Self-Consistency (3 Runs)
problem = (
    'Sales team: 8 people. 12 calls/day. 22 working days/month.\n'
    'Conversion: 4% calls become meetings. Close rate: 25%. Avg deal: Rs 1,50,000.\n\n'
    'Think step by step: What is the expected monthly revenue?'
)

for i in range(3):
    print(f'\n--- Run {i+1} ---')
    print(ask(problem, temperature=0.3))

print('\n' + '=' * 60)
print('SELF-CONSISTENCY: Do all 3 reach the SAME number?')
print('Correct: 8x12x22=2112 calls -> x4%=84.48 meetings -> x25%=21.12 deals -> x1,50,000 = Rs 31,68,000')

### Lab 8.3 — Self-Critique Prompting
**Task:** Generate → Critique → Improve — a 3-step quality loop.

In [ ]:
# Lab 8.3 — Self-Critique Loop
topic = 'Why companies should invest in employee upskilling programs'

# Step 1: Generate
draft = ask(f'Write a 150-word LinkedIn post about: {topic}. Hook + 3 points + engagement question.', temperature=0.6)
print('STEP 1 — DRAFT:'); print(draft)

# Step 2: Critique
critique = ask(
    f'Senior content strategist reviewing this LinkedIn post.\n\nPost: {draft}\n\n'
    f'Rate 1-5: Hook strength | Specificity | Engagement potential | Professional tone\n'
    f'Each: score + 1 improvement. Then: the ONE change that helps most.',
    temperature=0.3
)
print('\nSTEP 2 — CRITIQUE:'); print(critique)

# Step 3: Improve
improved = ask(
    f'Rewrite this LinkedIn post applying all feedback.\n\nOriginal: {draft}\n\nFeedback: {critique}\n\n'
    f'Improved version only. 150-200 words.',
    temperature=0.5
)
print('\nSTEP 3 — IMPROVED:'); print(improved)
print('\nKEY: Generate -> Critique -> Improve consistently produces better output than a single prompt.')

### Lab 8.4 — Prompt Decomposition
**Task:** Break a complex task into sequential steps.

In [ ]:
# Lab 8.4 — Prompt Decomposition: Raw Notes to Executive Summary
notes = (
    'New product: AI inventory forecasting for SME retailers. Launch Q1 2025.\n'
    'Target: 500 pilots in 6 months. Price: Rs 4,999/month.\n'
    'Works with existing POS. No IT team needed. Series A: 12Cr. Runway: 18 months.\n'
    'Competitor StockSense: 18-month head start, 2000 customers.\n'
    'Our advantage: 40% lower price, Hindi/regional language support.'
)

# Step 1: Extract
kp = ask(f'Extract 5 most strategically important points. Numbered. One sentence each.\n\n{notes}', temperature=0.2)
print('STEP 1 — KEY POINTS:'); print(kp)

# Step 2: Summarize
es = ask(
    f'Write a 150-word executive summary for the Board. 2 paragraphs: '
    f'(1) Opportunity and readiness (2) Risks and mitigation. Professional tone.\n\nPoints:\n{kp}',
    temperature=0.4
)
print('\nSTEP 2 — EXECUTIVE SUMMARY:'); show(es)

# Step 3: Polish
polished = ask(
    f'Edit: eliminate passive voice, replace jargon, strengthen opening sentence. '
    f'Output improved version then list 3 changes made.\n\n{es}',
    temperature=0.3
)
print('\nSTEP 3 — POLISHED:'); show(polished)

---
# SESSION 9 — Persona Prompting & Role Engineering
## Labs: Single Persona, Multi-Persona Panel, System Prompts

### Lab 9.1 — Same Question, Different Personas
**Task:** See how different expert personas produce fundamentally different answers.

In [ ]:
# Lab 9.1 — Multi-Persona Comparison
question = 'What are the biggest risks of adopting AI in a mid-size enterprise?'

personas = {
    'CTO': 'You are a CTO with 20 years of enterprise technology leadership. Focus on technical debt, integration, and infrastructure.',
    'CFO': 'You are a CFO focused on ROI, total cost of ownership, and financial risk management.',
    'CHRO': 'You are a Chief HR Officer focused on workforce impact, reskilling, and organizational change.',
    'Legal Counsel': 'You are General Counsel focused on regulatory compliance, IP, liability, and data protection.',
}

for role, persona in personas.items():
    print(f'\n{"=" * 50}')
    print(f'PERSONA: {role}')
    print('=' * 50)
    print(ask(f'{persona}\n\nAnswer in 100 words:\n{question}', temperature=0.5))

print('\nREFLECTION: Same question, 4 completely different perspectives.')
print('Persona engineering gives you expert-level thinking on demand.')

### Lab 9.2 — Expert Panel (Multi-Persona in One Prompt)
**Task:** Simulate a panel discussion with multiple experts in a single prompt.

In [ ]:
# Lab 9.2 — Expert Panel Discussion
panel = (
    'Facilitate an expert panel discussion. Three experts respond to the question below.\n\n'
    'PANELISTS:\n'
    '1. DR. MEERA NAIR — AI Ethics Professor. Focus: responsible AI, bias, societal impact.\n'
    '2. VIKRAM SHARMA — Tech Startup Founder. Focus: speed, practical application, competitive edge.\n'
    '3. PRIYA DESAI — Enterprise Risk Manager. Focus: governance, compliance, risk mitigation.\n\n'
    'QUESTION: Should companies adopt GenAI tools immediately or wait for clearer regulation?\n\n'
    'FORMAT:\n'
    '- Each expert: 60 words in their authentic voice\n'
    '- Then: MODERATOR SYNTHESIS (40 words): key agreement and key disagreement\n'
    '- Then: AUDIENCE TAKEAWAY (20 words): one actionable insight'
)

show(ask(panel, temperature=0.6, max_tokens=600), 'Expert Panel Discussion')

### Lab 9.3 — Engineered Persona with System Prompt
**Task:** Build a detailed persona using a system prompt for persistent behavior.

In [ ]:
# Lab 9.3 — Engineered System Prompt Persona
system_prompt = (
    'You are Sarah Chen, a Senior Marketing Strategist with 12 years of B2B SaaS experience.\n\n'
    'EXPERTISE: Content marketing, demand generation, brand positioning, marketing analytics.\n'
    'STYLE: Data-driven. Every recommendation includes a metric or benchmark.\n'
    'COMMUNICATION: Direct, structured, no fluff. Use frameworks when explaining.\n'
    'PERSONALITY: Confident but collaborative. Challenges assumptions constructively.\n\n'
    'RULES:\n'
    '- Never give vague advice like "create great content" — always be specific\n'
    '- Always ask clarifying questions if the request is too vague\n'
    '- Reference industry benchmarks when possible\n'
    '- If you don\'t know something, say so — never fabricate data'
)

# Test the persona with multiple questions
questions = [
    'How should we allocate our marketing budget next quarter? We have 15L.',
    'Our blog gets 5,000 monthly visitors but only 20 leads. What is wrong?',
    'Should we invest in influencer marketing for our B2B product?',
]

for q in questions:
    print(f'\nQ: {q}')
    print('-' * 60)
    print(ask(q, system=system_prompt, temperature=0.5))
    print()

print('REFLECTION: Does the persona stay consistent across all 3 answers?')
print('Check: data-driven, direct, no fluff, asks clarifying questions?')

### Lab 9.4 — Persona Stacking
**Task:** Stack two persona roles for a hybrid expert perspective.

In [ ]:
# Lab 9.4 — Persona Stacking
stacked = (
    'You combine the expertise of TWO professionals:\n'
    '1. A data scientist with 10 years of ML experience in financial services\n'
    '2. A financial compliance officer with deep knowledge of RBI regulations\n\n'
    'Question: A mid-size Indian NBFC wants to use AI for automated credit scoring. '
    'What approach should they take?\n\n'
    'Answer in 200 words combining BOTH perspectives. '
    'Structure: Technical Approach (from data scientist) | Compliance Requirements (from compliance officer) | Integrated Recommendation.'
)

show(ask(stacked, temperature=0.4), 'Persona Stacking — Data Science + Compliance')
print('\nKEY INSIGHT: Persona stacking produces uniquely valuable cross-functional advice.')

---
# SESSION 10 — Prompt Library & Optimization
## Labs: Optimization Cycle, Prompt Library Builder, Version Control

### Lab 10.1 — Prompt Optimization Cycle
**Task:** Take a basic prompt through 3 optimization iterations.

In [ ]:
# Lab 10.1 — 3-Version Optimization
article = (
    'India manufacturing PMI 56.4 (16-year high, Dec 2024). Autos and electronics led. Capacity: 82%.\n'
    'Input inflation +4.2% MoM. ICICI: margin pressure could slow Q1 2025 growth.'
)

v1 = f'Summarize this article.\n{article}'

v2 = (
    f'Financial analyst briefing investment committee.\n'
    f'Summarize in 3 bullets: What happened | Why it matters | Watch out for\n\n'
    f'Article: {article}'
)

v3 = (
    v2 + '\n\nAfter writing, self-evaluate: score Specificity (1-5) and Actionability (1-5). '
    'If either <4, rewrite that bullet to score 4+.'
)

for label, p in [('v1.0 Basic', v1), ('v2.0 Structured', v2), ('v3.0 Self-Critique', v3)]:
    print(f'\n{"=" * 50}')
    print(label)
    print('=' * 50)
    print(ask(p, temperature=0.3))

print('\nOPTIMIZATION PATTERN: Basic -> Add role+format -> Add self-evaluation')

### Lab 10.2 — Build Your Professional Prompt Library
**Task:** Create a versioned, searchable prompt library with metadata.

In [ ]:
# Lab 10.2 — Professional Prompt Library
library = {}

def add_prompt(pid, title, category, technique, version, prompt_text, perf, variables):
    library[pid] = {
        'id': pid, 'title': title, 'category': category, 'technique': technique,
        'version': version, 'prompt': prompt_text, 'performance': perf,
        'variables': variables, 'created': str(datetime.date.today())
    }
    print(f'Added: [{pid}] {title} ({version})')

def get_prompt(pid, **kwargs):
    if pid not in library: return f'Not found: {pid}'
    t = library[pid]['prompt']
    for k, v in kwargs.items(): t = t.replace(f'{{{k}}}', v)
    return t

def list_library():
    print(f'{"ID":<12} {"Title":<35} {"Ver":<8} {"Category"}')
    print('-' * 70)
    for e in library.values():
        print(f'{e["id"]:<12} {e["title"]:<35} {e["version"]:<8} {e["category"]}')

# Add prompts to the library
add_prompt('MKT-001', 'LinkedIn Post Generator', 'marketing', 'persona+few-shot', 'v2.1',
    'You are {ROLE}. Brand voice: {BRAND_VOICE}. Topic: {TOPIC}.\n'
    'Write 200-250 word LinkedIn post. Hook: specific insight. Body: 3 points. CTA: open question.\n'
    'Do NOT start with I, use "excited to share" or "game-changing".',
    {'success_rate': '87%', 'tested': 45}, ['ROLE', 'BRAND_VOICE', 'TOPIC'])

add_prompt('HR-001', 'Inclusive JD Generator', 'hr', 'craft+constraints', 'v1.3',
    'Senior HRBP. Write inclusive JD for {ROLE}. Must-haves: {MUST_HAVE}. Nice-to-have: {NICE}.\n'
    'Lead with impact. No unnecessary degree requirements. Separate Must/Nice-to-Have.',
    {'success_rate': '91%', 'tested': 30}, ['ROLE', 'MUST_HAVE', 'NICE'])

add_prompt('CS-001', 'Customer Response (HEARD)', 'support', 'few-shot+HEARD', 'v1.0',
    'Customer support specialist using HEARD framework.\n'
    'H=Hear (acknowledge), E=Empathize, A=Apologize, R=Resolve (action+timeline), D=Diagnose (prevent).\n'
    'Complaint: {COMPLAINT}\nCustomer: {CUSTOMER}\nUnder 150 words. Warm, specific, accountable.',
    {'success_rate': '85%', 'tested': 20}, ['COMPLAINT', 'CUSTOMER'])

add_prompt('FIN-001', 'Variance Commentary', 'finance', 'craft+CoT', 'v1.1',
    'FP&A analyst writing board-level variance commentary. Plain English.\n'
    '2 sentences: (1) net variance + specific driver (2) offsetting factor + magnitude.\n'
    'Data: {VARIANCE_DATA}\nDo NOT use: market conditions, various factors. Name specifics.',
    {'success_rate': '89%', 'tested': 25}, ['VARIANCE_DATA'])

add_prompt('RPT-001', 'Executive Summary Generator', 'reporting', 'decomposition', 'v1.2',
    'Senior consultant. Write 150-word executive summary for {AUDIENCE}.\n'
    'Topic: {TOPIC}\nKey points: {KEY_POINTS}\n'
    '2 paragraphs: (1) Situation + key finding (2) Recommendation + next step.\n'
    'Professional, evidence-based. Active voice only.',
    {'success_rate': '88%', 'tested': 35}, ['AUDIENCE', 'TOPIC', 'KEY_POINTS'])

print('\n'); list_library()

### Lab 10.3 — Using Your Prompt Library
**Task:** Retrieve prompts from the library, fill variables, and generate output.

In [ ]:
# Lab 10.3 — Library Usage

# Use MKT-001 with variables filled
filled = get_prompt('MKT-001',
    ROLE='VP Marketing at a B2B SaaS company',
    BRAND_VOICE='Direct, evidence-based, no buzzwords',
    TOPIC='Why vanity metrics kill marketing strategy'
)
show(ask(filled, temperature=0.6), 'Library Output: MKT-001')

print('\n' + '=' * 60 + '\n')

# Use CS-001
filled2 = get_prompt('CS-001',
    COMPLAINT='I was promised a callback within 2 hours but nobody called. It has been 3 days.',
    CUSTOMER='Deepak'
)
show(ask(filled2, temperature=0.4), 'Library Output: CS-001')

print('\n' + '=' * 60 + '\n')

# Use FIN-001
filled3 = get_prompt('FIN-001',
    VARIANCE_DATA='Sep 2024: Actual 42.3Cr, Plan 38.0Cr, +4.3Cr (+11.3%). Driver: Enterprise closed 3 deals early. SMB missed 0.8Cr — 2 deals slipped to Oct.'
)
show(ask(filled3, temperature=0.2), 'Library Output: FIN-001')

print('\nPROMPT LIBRARY BENEFITS:')
print('  1. Consistency — same quality every time')
print('  2. Speed — no rewriting from scratch')
print('  3. Versioning — track what works')
print('  4. Shareability — team uses the same proven prompts')

### Lab 10.4 — Module 2 Capstone: Build & Optimize a Complete Prompt
**Task:** Build a production-ready prompt from scratch, score it, and add it to your library.

In [ ]:
# Lab 10.4 — Module 2 Capstone
# YOUR TASK: Build a production-ready prompt for a task relevant to YOUR work.
# Follow these steps:

# Step 1: Define the prompt using CRAFT
my_prompt = craft_prompt(
    context='[REPLACE: Describe your specific professional context]',
    role='[REPLACE: The expert persona needed]',
    action='[REPLACE: The specific task with clear deliverable]',
    fmt='[REPLACE: Exact output structure — sections, length, format type]',
    tone='[REPLACE: Specific tone + what to avoid]',
    constraints='[REPLACE: Guardrails, limits, edge case handling]'
)

# Step 2: Test it
print('YOUR PROMPT OUTPUT:')
output = ask(my_prompt, temperature=0.4)
show(output)

# Step 3: Score it using CRAFT audit
audit = (
    f'Score this prompt on CRAFT (each 1-5). Markdown table. Total /30.\n'
    f'C (Context) | R (Role) | A (Action) | F (Format) | T (Tone) | Constraints\n\n'
    f'Prompt: "{my_prompt}"'
)
print('\nCRAFT SCORE:')
show(ask(audit, temperature=0.2))

# Step 4: If score < 26, optimize and re-score
print('\nNEXT STEPS:')
print('1. If score < 26/30: identify the weakest CRAFT element and improve it')
print('2. Test with 3 different realistic inputs')
print('3. Add to your prompt library using add_prompt()')
print('\n' + '=' * 60)
print('MODULE 2 COMPLETE')
print('You now know: CRAFT, Zero/Few-Shot, CoT, Self-Critique, Persona Engineering, Prompt Libraries')
print('Next: Module 3 — AI for Productivity (Sessions 11-15)')